# Lección 02 — Enfoque 2: Clase Agente Personalizada

En la lección 01 escribiste el bucle agente de forma manual. En este notebook vamos a **encapsular ese bucle en una clase reutilizable**.

La ventaja: una vez que tenés la clase, crear un nuevo agente es cuestión de 5 líneas de código.

### Lo que vas a aprender:
1. Cómo crear una clase `Agente` reutilizable
2. Cómo agregar **memoria de conversación** (multi-turno)
3. Cómo crear diferentes agentes usando la misma clase base

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

## La Clase Agente

Esta clase encapsula todo lo que hacías manualmente en la lección 01:
- El bucle de razonar → actuar → observar
- La gestión del historial de mensajes (memoria)
- El registro y ejecución de herramientas

Una vez que la tenés, crear un agente nuevo es así de simple:
```python
mi_agente = Agente(nombre="Asistente", instrucciones="Sos un...", herramientas=[mi_funcion])
```

In [ ]:
class Agente:
    """Clase base para crear agentes de IA con Claude."""
    
    def __init__(self, nombre: str, instrucciones: str, herramientas: list = None, modelo: str = "claude-opus-4-5"):
        self.nombre = nombre
        self.instrucciones = instrucciones
        self.modelo = modelo
        self.herramientas_func = {}  # nombre → función Python
        self.herramientas_schema = []  # schemas para Claude
        self.historial = []  # memoria de conversación
        
        # Registrar las herramientas si se proporcionaron
        if herramientas:
            for h in herramientas:
                self.agregar_herramienta(h["schema"], h["funcion"])
    
    def agregar_herramienta(self, schema: dict, funcion):
        """Registra una herramienta en el agente."""
        self.herramientas_schema.append(schema)
        self.herramientas_func[schema["name"]] = funcion
    
    def limpiar_memoria(self):
        """Borra el historial de conversación (empieza una sesión nueva)."""
        self.historial = []
        print(f"[{self.nombre}] Memoria limpiada.")
    
    def hablar(self, mensaje: str, verbose: bool = True) -> str:
        """Envía un mensaje al agente y obtiene su respuesta."""
        
        # Agregar el mensaje del usuario al historial
        self.historial.append({"role": "user", "content": mensaje})
        
        if verbose:
            print(f"Vos: {mensaje}")
            print("-" * 50)
        
        # Bucle agente
        while True:
            respuesta = client.messages.create(
                model=self.modelo,
                max_tokens=1024,
                system=self.instrucciones,
                tools=self.herramientas_schema if self.herramientas_schema else anthropic.NOT_GIVEN,
                messages=self.historial
            )
            
            # ¿El agente quiere usar una herramienta?
            if respuesta.stop_reason == "tool_use":
                uso_tool = next(b for b in respuesta.content if b.type == "tool_use")
                
                if verbose:
                    print(f"[{self.nombre} usa: {uso_tool.name}]")
                
                # Ejecutar la herramienta
                funcion = self.herramientas_func.get(uso_tool.name)
                if funcion:
                    resultado = funcion(**uso_tool.input) if uso_tool.input else funcion()
                else:
                    resultado = f"Error: herramienta '{uso_tool.name}' no encontrada"
                
                # Agregar al historial y continuar
                self.historial.append({"role": "assistant", "content": respuesta.content})
                self.historial.append({
                    "role": "user",
                    "content": [{
                        "type": "tool_result",
                        "tool_use_id": uso_tool.id,
                        "content": json.dumps(resultado, ensure_ascii=False)
                    }]
                })
                continue
            
            # Respuesta final
            texto_final = next(b.text for b in respuesta.content if b.type == "text")
            self.historial.append({"role": "assistant", "content": texto_final})
            
            if verbose:
                print(f"{self.nombre}: {texto_final}")
            
            return texto_final

print("Clase Agente definida.")

## Creando el Agente de Viajes

Ahora creamos el mismo agente de la lección 01, pero usando nuestra clase. Fijate lo limpio que queda el código.

In [ ]:
# Definir la función de la herramienta
def verificar_disponibilidad(destino: str) -> dict:
    """Verifica si un destino está disponible para reservar."""
    disponibilidad = {
        "Barcelona": {"disponible": True, "precio_promedio": "USD 1200", "mejor_epoca": "mayo-octubre"},
        "Tokio": {"disponible": True, "precio_promedio": "USD 1800", "mejor_epoca": "marzo-mayo"},
        "Cancún": {"disponible": True, "precio_promedio": "USD 900", "mejor_epoca": "todo el año"},
        "Cartagena": {"disponible": True, "precio_promedio": "USD 700", "mejor_epoca": "diciembre-abril"},
        "Bali": {"disponible": False, "motivo": "temporada alta agotada"},
        "Dubai": {"disponible": False, "motivo": "sin vuelos directos disponibles"},
    }
    return disponibilidad.get(destino, {"disponible": False, "motivo": "destino no encontrado en nuestro catálogo"})

# Schema de la herramienta para Claude
schema_disponibilidad = {
    "name": "verificar_disponibilidad",
    "description": "Verifica si un destino de viaje está disponible para reservar y obtiene información de precios.",
    "input_schema": {
        "type": "object",
        "properties": {
            "destino": {
                "type": "string",
                "description": "El nombre del destino a verificar (ej: Barcelona, Tokio, Cancún)"
            }
        },
        "required": ["destino"]
    }
}

# Crear el agente — 5 líneas de código
agente_viajes = Agente(
    nombre="TravelBot",
    instrucciones="""Sos un agente experto en viajes para latinoamericanos. 
    Siempre verificá la disponibilidad antes de recomendar un destino. 
    Respondé en español, de forma amigable y con información útil sobre precios y época de viaje.""",
    herramientas=[{"schema": schema_disponibilidad, "funcion": verificar_disponibilidad}]
)

print("Agente de viajes creado.")

## Conversación Multi-turno

La gran ventaja de nuestra clase es que **recuerda el contexto** entre mensajes. El agente sabe lo que se habló antes en la conversación.

In [ ]:
# Primer mensaje
agente_viajes.hablar("Quiero ir a Barcelona, ¿está disponible?")

In [ ]:
# El agente recuerda que estábamos hablando de Barcelona
agente_viajes.hablar("¿Y cuál es la mejor época para ir?")

In [ ]:
# Pedir otra opción
agente_viajes.hablar("¿Tenés algo más económico en Latinoamérica?")

## Creando un Segundo Agente — Mismo molde, diferente propósito

Ahora vas a ver el poder real de tener una clase: en segundos creamos un agente completamente diferente reutilizando el mismo código.

In [ ]:
# Un agente para responder preguntas sobre IA — sin herramientas
agente_ia = Agente(
    nombre="ProfeIA",
    instrucciones="""Sos un profesor de IA para principiantes latinoamericanos. 
    Explicás conceptos técnicos de forma simple, usando analogías del día a día. 
    Nunca usás jerga técnica sin explicarla. Respondés en español rioplatense."""
)

agente_ia.hablar("¿Qué es exactamente un token en IA?")

In [ ]:
agente_ia.hablar("¿Y por qué importa la cantidad de tokens?")

## Resumen

Construiste una clase `Agente` reutilizable que:

- **Encapsula el bucle agente** — no tenés que escribirlo cada vez
- **Gestiona la memoria** — recuerda el historial de conversación automáticamente
- **Registra herramientas** — podés agregar funciones fácilmente
- **Es reutilizable** — el mismo molde para cualquier tipo de agente

### Comparación de enfoques:

| | SDK Directo (L01) | Clase Agente (L02) |
|---|---|---|
| Código para crear agente | ~40 líneas | ~5 líneas |
| Memoria automática | No | Sí |
| Reutilizable | No | Sí |
| Control fino | Total | Total |

---

En la **Lección 03** vamos a aprender los patrones de diseño más comunes — las "recetas" que los expertos usan para resolver problemas con agentes.